# 📦 Export to GGUF for Local Use
> Convert your fine-tuned model to GGUF format for llama.cpp / Ollama.

**Input:** Fine-tuned model from Stage 2 (or Stage 1)
**Output:** Quantized GGUF file ready for `llama-cli` or `ollama`


In [ ]:
!pip install -q huggingface_hub
!git clone https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!cd /tmp/llama.cpp && pip install -q -r requirements.txt

In [ ]:
import os
import subprocess

# Configuration
MODEL_PATH = "outputs/stage2_tool_calling/merged_model"
OUTPUT_NAME = "code-llm-forge-v1"
QUANT_METHODS = ["Q4_K_M", "Q5_K_M", "Q8_0"]  # Common sweet spots

if not os.path.exists(MODEL_PATH):
    MODEL_PATH = "outputs/stage1_code_reasoning/merged_model"
    OUTPUT_NAME = "code-llm-forge-stage1"

if not os.path.exists(MODEL_PATH):
    print("❌ No fine-tuned model found. Run training notebooks first.")
    print("   Using base model for demo...")
    MODEL_PATH = "Qwen/Qwen2.5-Coder-7B-Instruct"
    OUTPUT_NAME = "qwen25-coder-7b-demo"

os.makedirs("gguf_output", exist_ok=True)
print(f"📥 Model: {MODEL_PATH}")
print(f"📤 Output: {OUTPUT_NAME}")

In [ ]:
# Step 1: Convert to GGUF FP16
print("🔄 Converting to GGUF (FP16)...")
cmd = f"""python /tmp/llama.cpp/convert_hf_to_gguf.py {MODEL_PATH} \
    --outfile gguf_output/{OUTPUT_NAME}-f16.gguf \
    --outtype f16"""

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
if result.returncode == 0:
    print("✅ FP16 GGUF created")
else:
    print(f"❌ Error: {result.stderr[:300]}")

In [ ]:
# Step 2: Quantize to various formats
# Build llama.cpp quantize tool
print("🔨 Building quantize tool...")
subprocess.run("cd /tmp/llama.cpp && cmake -B build && cmake --build build --target llama-quantize -j4",
               shell=True, capture_output=True)

for quant in QUANT_METHODS:
    print(f"\n🔄 Quantizing to {quant}...")
    cmd = f"/tmp/llama.cpp/build/bin/llama-quantize gguf_output/{OUTPUT_NAME}-f16.gguf gguf_output/{OUTPUT_NAME}-{quant}.gguf {quant}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        size_mb = os.path.getsize(f"gguf_output/{OUTPUT_NAME}-{quant}.gguf") / (1024*1024)
        print(f"  ✅ {quant}: {size_mb:.0f} MB")
    else:
        print(f"  ❌ Error: {result.stderr[:200]}")

In [ ]:
# Summary
print("\n" + "=" * 60)
print("📦 GGUF EXPORT SUMMARY")
print("=" * 60)

for f in sorted(os.listdir("gguf_output")):
    if f.endswith(".gguf"):
        size = os.path.getsize(f"gguf_output/{f}") / (1024*1024)
        print(f"  {f}: {size:,.0f} MB")

print(f"""
\n🚀 To run locally:

  # Interactive chat
  llama-cli -m gguf_output/{OUTPUT_NAME}-Q8_0.gguf \\
    -co -cnv -fa -ngl 99 -c 8192 \\
    -p "You are a helpful coding assistant with tool-calling capabilities."

  # OpenAI-compatible API server
  llama-server -m gguf_output/{OUTPUT_NAME}-Q8_0.gguf \\
    -ngl 99 -c 8192 --port 8080

  # Or import into Ollama
  # Create a Modelfile pointing to your GGUF, then:
  # ollama create code-forge -f Modelfile
""")